# Road Damage Severity Index (SI)
This notebook will demonstrate how to calculate a Road Severity Index using YOLOv8 inference outputs.

## 1. YOLOv8 Inference Output Structure
When we run `results = model.predict(image)`, YOLOv8 returns a list of `Results` objects one for each input image.

The `Results` object contains different attributes depending on the task. For object detection, the primary attribute is `results.boxes`, which is a specialized `Boxes` object containing all detected bounding boxes. 

Inside `results.boxes`, I can extract the following:
- `boxes.xyxy`: This bounds box coordinates in `[x1, y1, x2, y2]` format.
- `boxes.conf`: This gives the confidence score.
- `boxes.cls`: Basically gives the class ID for the detected object.

## 2. Severity Index Formula

To calculate a Severity Index (SI) that quantifies the overall damage on a road given a single frame, we need to balance the severity of the damage type, the model's confidence, and the physical footprint of the damage relative to the camera view.

**Weights ($W$)**
- Potholes: `1.0` indicates highest sevearity
- Alligator Cracks: `0.8` High sevearity
- Longitudinal Cracks: `0.5` Medium severity
- Transverse Cracks: `0.3` Low severity

**Formula**
The formula for severity index that I am going to be using is:
$$SI = \sum_{i=1}^{n} (W_i \times Confidence_i \times A_{rel, i})$$

Where $A_{rel}$ is the bounding box area relative to the total frame area.

In [1]:
import numpy as np
def calculate_severity_index(results, frame_area, weights=None):
    
    if weights is None:
        #Indexed as 0: Potholes, 1: Alligator, 2: Longitudinal, 3: Transverse
        weights = { 0: 1.0, 1: 0.8, 2: 0.5, 3: 0.3 }
        
    severity_index = 0.0
    
    # Checking for detection
    if results.boxes is None or len(results.boxes) == 0:
        return severity_index
        
    boxes = results.boxes
    
    # Array extraction
    coords = boxes.xyxy.cpu().numpy() 
    confidences = boxes.conf.cpu().numpy()
    class_ids = boxes.cls.cpu().numpy()
    
    for i in range(len(boxes)):
        x1, y1, x2, y2 = coords[i]
        
        # Bounding box area and relative area calculation
        bbox_area = (x2 - x1) * (y2 - y1)
        relative_area = bbox_area / frame_area
        
        # Finally calculate the Severity Index 
        weight = weights.get(int(class_ids[i]), 0.1)
        severity_index += weight * confidences[i] * relative_area
        
    return severity_index

In [2]:
#  Mock Testing 
from ultralytics import YOLO
import cv2
import torch
import numpy as np

print("Loading model")
model = YOLO("yolov8n.pt") 
dummy_img = np.zeros((480, 640, 3), dtype=np.uint8)

print("Running model prediction")
results_list = model.predict(dummy_img, verbose=False)
results = results_list[0]

# Mock detections for testing
class MockBoxes:
    def __len__(self): return 2
    
results.boxes = MockBoxes()
results.boxes.xyxy = type('MockTensor', (), {'cpu': lambda self: type('MockCPU', (), {'numpy': lambda self: np.array([[100.0, 100.0, 200.0, 200.0], [300.0, 300.0, 350.0, 350.0]])})()})()
results.boxes.conf = type('MockTensor', (), {'cpu': lambda self: type('MockCPU', (), {'numpy': lambda self: np.array([0.95, 0.82])})()})()
results.boxes.cls = type('MockTensor', (), {'cpu': lambda self: type('MockCPU', (), {'numpy': lambda self: np.array([0, 1])})()})()

print("\nExtraction Example:")
coords = results.boxes.xyxy.cpu().numpy()
confidences = results.boxes.conf.cpu().numpy()
class_ids = results.boxes.cls.cpu().numpy()

for i in range(len(coords)):
    x1, y1, x2, y2 = coords[i]
    print(f"Detection {i+1}: BBox ({x1:.1f}, {y1:.1f}, {x2:.1f}, {y2:.1f}), Class ID: {int(class_ids[i])}, Conf: {confidences[i]:.4f}")
    
frame_width, frame_height = 640, 480
total_area = frame_width * frame_height

si = calculate_severity_index(results, total_area)
print(f"\nFrame Area: {total_area} px^2")
print(f"Calculated Severity Index: {si:.5f}")

Loading model
Running model prediction

Extraction Example:
Detection 1: BBox (100.0, 100.0, 200.0, 200.0), Class ID: 0, Conf: 0.9500
Detection 2: BBox (300.0, 300.0, 350.0, 350.0), Class ID: 1, Conf: 0.8200

Frame Area: 307200 px^2
Calculated Severity Index: 0.03626
